In [1]:
# ==========================================================
# BLOCK 1: BINARY SETUP & DATA LOADING
# ==========================================================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# 1. Global Reproducibility (Q1 Journal Requirement)
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 2. GPU Setup (Safe Memory Growth)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPUs Detected: {len(gpus)}")
    except RuntimeError as e:
        print(e)

# 3. Directories & Constants
DATASET_ROOT = '/home/T2430514/Downloads/MargeDataset/Binary'
ANOMALY_DIR = os.path.join(DATASET_ROOT, 'Anomaly')
NORMAL_DIR = os.path.join(DATASET_ROOT, 'Normal')
PROCESSED_DATA_DIR = '/home/T2430514/Downloads/MargeDataset/Processed' 
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

FRAME_SIZE = (224, 224)
NUM_FRAMES = 16
BATCH_SIZE = 4 

# 4. Data Loading Logic
def create_dataframe():
    data = []
    # Anomaly (Label 1)
    if os.path.exists(ANOMALY_DIR):
        for video_file in os.listdir(ANOMALY_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(ANOMALY_DIR, video_file), 'bin_label': 1})
    
    # Normal (Label 0)
    if os.path.exists(NORMAL_DIR):
        for video_file in os.listdir(NORMAL_DIR):
            if video_file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                data.append({'path': os.path.join(NORMAL_DIR, video_file), 'bin_label': 0})
                
    return pd.DataFrame(data)

all_df = create_dataframe()
print(f"Total Binary Videos: {len(all_df)}")

# 5. Stratified Split (Crucial for Imbalanced/Small Data)
train_df, temp_df = train_test_split(all_df, test_size=0.2, stratify=all_df['bin_label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['bin_label'], random_state=SEED)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# 6. Compute Class Weights
y_train = train_df['bin_label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))
print(f"Class Weights: {class_weights_dict}")

2026-06-09 23:33:24.273907: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-09 23:33:24.313574: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781026404.325048 1684813 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781026404.329281 1684813 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781026404.367794 1684813 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

GPUs Detected: 1
Total Binary Videos: 4534
Train: 3627 | Val: 453 | Test: 454
Class Weights: {0: np.float64(1.0002757859900717), 1: np.float64(0.9997243660418964)}


In [2]:
# ==========================================================
# BLOCK 2: FRAME EXTRACTION
# ==========================================================
import sys

def extract_and_save_frames(dataframe, output_dir, num_frames=NUM_FRAMES, frame_size=FRAME_SIZE):
    print(f"Processing {len(dataframe)} videos...")
    count = 0
    
    for idx, row in dataframe.iterrows():
        base_name = os.path.basename(row.path)
        # Use .npy for faster loading during training
        save_name = os.path.splitext(base_name)[0] + '.npy'
        output_path = os.path.join(output_dir, save_name)
        
        # Skip if already processed
        if os.path.exists(output_path): 
            continue

        cap = cv2.VideoCapture(row.path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            cap.release()
            continue
            
        # Uniform Temporal Sampling (SOTA standard)
        frame_indices = np.linspace(0, max(total_frames - 1, 0), num=num_frames, dtype=int)
        frames = []
        
        for i in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, frame_size)
                frames.append(frame)
            else:
                # Padding if read fails
                frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
        cap.release()
        
        # Ensure exact frame count
        while len(frames) < num_frames:
            frames.append(np.zeros(frame_size + (3,), dtype=np.uint8))
            
        # Save as uint8 to save disk space (converted to float32 in generator)
        np.save(output_path, np.array(frames, dtype=np.uint8))
        
        count += 1
        if count % 100 == 0: 
            sys.stdout.write(f"\rExtracted {count} videos.")
    print("\nExtraction Complete.")

extract_and_save_frames(all_df, PROCESSED_DATA_DIR)

Processing 4534 videos...

Extraction Complete.


In [3]:
# ==========================================================
# BLOCK 3: SOTA DATA GENERATOR (EXPERIMENT C: CUTMIX)
# ==========================================================
import tensorflow as tf
import numpy as np
import os

class CutMixVideoDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, processed_data_dir, batch_size=BATCH_SIZE, 
                 num_frames=NUM_FRAMES, frame_size=FRAME_SIZE, 
                 augment=False, shuffle=True, cutmix_alpha=1.0):
        self.dataframe = dataframe
        self.processed_data_dir = processed_data_dir
        self.batch_size = batch_size
        self.num_frames = num_frames
        self.frame_size = frame_size
        self.augment = augment
        self.shuffle = shuffle
        self.cutmix_alpha = cutmix_alpha
        self.indices = np.arange(len(self.dataframe))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.dataframe) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        return self.__data_generation(batch_indices)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def load_video(self, video_path):
        base_name = os.path.basename(video_path)
        name, _ = os.path.splitext(base_name)
        npy_path = os.path.join(self.processed_data_dir, name + '.npy')
        
        if os.path.exists(npy_path):
            try: 
                return np.load(npy_path).astype(np.float32) / 255.0
            except: 
                pass
        return np.zeros((self.num_frames, *self.frame_size, 3), dtype=np.float32)

    def rand_bbox(self, size, lam):
        H, W = size
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)

        cx = np.random.randint(W)
        cy = np.random.randint(H)

        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)

        return bbx1, bby1, bbx2, bby2

    def apply_consistent_augmentation(self, video):
        """
        Applies Memory-Safe Augmentations using pure NumPy.
        Prevents TensorFlow EagerTensor RAM leaks.
        """
        # 1. Random Horizontal Flip (axis=2 is the width dimension)
        if np.random.rand() > 0.5:
            video = np.flip(video, axis=2)
            
        # 2. Random Brightness (-0.15 to 0.15)
        brightness_delta = np.random.uniform(-0.15, 0.15)
        video = video + brightness_delta
        
        # 3. Random Contrast (0.85 to 1.15)
        contrast_factor = np.random.uniform(0.85, 1.15)
        mean = np.mean(video, axis=(1, 2, 3), keepdims=True)
        video = (video - mean) * contrast_factor + mean
        
        # Clip strictly to [0.0, 1.0] and ensure float32
        return np.clip(video, 0.0, 1.0).astype(np.float32)

    def __data_generation(self, batch_indices):
        X = np.empty((self.batch_size, self.num_frames, *self.frame_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size), dtype=np.float32)

        for i, idx in enumerate(batch_indices):
            row = self.dataframe.iloc[idx]
            video = self.load_video(row.path)
            label = float(row.bin_label)

            # --- SOTA: TUBE CUTMIX LOGIC ---
            if self.augment and self.cutmix_alpha > 0 and np.random.rand() < 0.5:
                rand_idx = np.random.choice(self.indices)
                row2 = self.dataframe.iloc[rand_idx]
                video2 = self.load_video(row2.path)
                label2 = float(row2.bin_label)

                lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
                bbx1, bby1, bbx2, bby2 = self.rand_bbox(self.frame_size, lam)
                
                # Tube CutMix (applies patch identically through all frames)
                video[:, bby1:bby2, bbx1:bbx2, :] = video2[:, bby1:bby2, bbx1:bbx2, :]
                
                # Adjust label based on patch area
                patch_ratio = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (self.frame_size[0] * self.frame_size[1]))
                label = patch_ratio * label + (1 - patch_ratio) * label2

            # --- SPATIAL AUGMENTATION ---
            if self.augment:
                video = self.apply_consistent_augmentation(video)

            X[i,] = video
            y[i] = label

        return X, y
    
    def get_labels(self):
        original_indices = self.indices.copy()
        if self.shuffle:
            sorted_indices = np.arange(len(self.dataframe))
        else:
            sorted_indices = self.indices
            
        limit = self.__len__() * self.batch_size
        return self.dataframe.iloc[sorted_indices[:limit]]['bin_label'].values

# Alias for model blocks to use seamlessly
SOTAVideoDataGenerator = CutMixVideoDataGenerator

print("Initializing Generators (Experiment C: CutMix Augmentation - NUMPY FIX)...")
train_generator = SOTAVideoDataGenerator(
    train_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=True, cutmix_alpha=1.0, shuffle=True
)
val_generator = SOTAVideoDataGenerator(
    val_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False
)
print("CutMix Generators Ready.")

Initializing Generators (Experiment C: CutMix Augmentation - NUMPY FIX)...
CutMix Generators Ready.


In [4]:
# ==========================================================
# BLOCK 4 & 5: 3D RESNET MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
clear_session()
gc.collect()

# 2. SOTA Training Configuration
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define 3D ResNet Building Blocks
def conv3d_bn(x, filters, kernel_size, strides=(1,1,1), padding='same', activation=True):
    x = Conv3D(filters, kernel_size, strides=strides, padding=padding, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    if activation:
        x = Activation('relu')(x)
    return x

def resnet_block(x, filters, strides=(1,1,1)):
    shortcut = x
    x = conv3d_bn(x, filters, kernel_size=(3,3,3), strides=strides)
    x = conv3d_bn(x, filters, kernel_size=(3,3,3), activation=False)
    if strides != (1,1,1) or x.shape[-1] != shortcut.shape[-1]:
        shortcut = conv3d_bn(shortcut, filters, kernel_size=(1,1,1), strides=strides, activation=False)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_resnet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    x = conv3d_bn(video_input, 64, kernel_size=(7,7,7), strides=(1,2,2))
    x = tf.keras.layers.MaxPooling3D(pool_size=(1,3,3), strides=(1,2,2), padding='same')(x)
    
    x = resnet_block(x, 64)
    x = resnet_block(x, 64)
    x = resnet_block(x, 128, strides=(2,2,2)) 
    x = resnet_block(x, 128)
    x = resnet_block(x, 256, strides=(2,2,2))
    x = resnet_block(x, 256)
    x = resnet_block(x, 512, strides=(2,2,2))
    x = resnet_block(x, 512)
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)
    return Model(inputs=video_input, outputs=output, name='ResNet3D_18')

# 5. Initialize & Compile
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resnet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

# 6. Start Training
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 7. SOTA Evaluation & Inference Benchmarking
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

# Inference Latency/Throughput tracking
inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

# Specificity (True Negative Rate)
spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

# Parameter Calculations & FP32 Model Size
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 8. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

I0000 00:00:1777079502.918682 2249805 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13579 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: ResNet3D_18...
Epoch 1/50


I0000 00:00:1777079507.618987 2249906 service.cc:152] XLA service 0x7dffc0845500 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777079507.619007 2249906 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-25 07:11:47.832193: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777079508.764313 2249906 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-25 07:11:52.830154: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller b

  1/906 ━━━━━━━━━━━━━━━━━━━━ 3:35:21 14s/step - accuracy: 0.2500 - auc: 0.2500 - loss: 1.6574

2026-04-25 07:11:57.634430: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion', 484 bytes spill stores, 484 bytes spill loads

I0000 00:00:1777079517.677808 2249906 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 89s 83ms/step - accuracy: 0.5063 - auc: 0.6956 - loss: 0.7814 - val_accuracy: 0.5000 - val_auc: 0.4823 - val_loss: 32.4679
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 73s 81ms/step - accuracy: 0.5624 - auc: 0.7591 - loss: 0.6815 - val_accuracy: 0.7876 - val_auc: 0.8650 - val_loss: 0.5639
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 73s 81ms/step - accuracy: 0.5944 - auc: 0.7794 - loss: 0.6455 - val_accuracy: 0.7765 - val_auc: 0.8732 - val_loss: 0.6060
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 73s 80ms/step - accuracy: 0.5781 - auc: 0.7893 - loss: 0.6360 - val_accuracy: 0.6372 - val_auc: 0.8238 - val_loss: 0.8787
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 73s 81ms/step - accuracy: 0.5971 - auc: 0.8109 - loss: 0.6131 - val_accuracy: 0.8142 - val_auc: 0.8936 - val_loss: 0.5378
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 73s 80ms/step - accuracy: 0.6095 - auc: 0.8211 - loss: 0.5934 - val_accuracy: 0.7942 - val_auc: 0.8932 - val_loss: 0.5273
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 6 & 7: I3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for I3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define I3D Building Blocks
# ----------------------------------------------------------
def conv3d_bn(x, filters, kernel_size, padding='same', strides=(1,1,1), name=None):
    x = Conv3D(filters, kernel_size, strides=strides, padding=padding, 
               use_bias=False, kernel_regularizer=l2(1e-5), name=name)(x)
    x = BatchNormalization(scale=False)(x)
    x = Activation('relu')(x)
    return x

def inception_module(x, filters):
    f1x1, f3x3_reduce, f3x3, f5x5_reduce, f5x5, f_pool = filters

    branch1 = conv3d_bn(x, f1x1, (1, 1, 1))
    
    branch2 = conv3d_bn(x, f3x3_reduce, (1, 1, 1))
    branch2 = conv3d_bn(branch2, f3x3, (3, 3, 3))

    branch3 = conv3d_bn(x, f5x5_reduce, (1, 1, 1))
    branch3 = conv3d_bn(branch3, f5x5, (3, 3, 3))

    branch4 = MaxPooling3D((3, 3, 3), strides=(1, 1, 1), padding='same')(x)
    branch4 = conv3d_bn(branch4, f_pool, (1, 1, 1))

    x = Concatenate()([branch1, branch2, branch3, branch4])
    return x

def create_i3d_model(input_shape):
    video_input = Input(shape=input_shape)

    x = conv3d_bn(video_input, 64, (7, 7, 7), strides=(2, 2, 2))
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)
    
    x = conv3d_bn(x, 64, (1, 1, 1))
    x = conv3d_bn(x, 192, (3, 3, 3))
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = inception_module(x, [64, 96, 128, 16, 32, 32])
    x = inception_module(x, [128, 128, 192, 32, 96, 64])
    x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2), padding='same')(x)

    x = inception_module(x, [192, 96, 208, 16, 48, 64])
    x = inception_module(x, [160, 112, 224, 24, 64, 64])
    x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2), padding='same')(x)

    x = inception_module(x, [128, 128, 256, 24, 64, 64])
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='I3D_Inception')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_i3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50 
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_i3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

# Parameter Calculations & FP32 Model Size
total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for I3D.

Training Model: I3D_Inception...
Epoch 1/50
905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.4688 - auc: 0.6617 - loss: 0.7640

2026-04-25 08:02:39.714635: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1045', 96 bytes spill stores, 96 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 60s 47ms/step - accuracy: 0.4994 - auc: 0.6916 - loss: 0.7399 - val_accuracy: 0.6460 - val_auc: 0.8036 - val_loss: 0.6950
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5494 - auc: 0.7600 - loss: 0.6830 - val_accuracy: 0.7699 - val_auc: 0.8611 - val_loss: 0.5972
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5756 - auc: 0.7921 - loss: 0.6606 - val_accuracy: 0.7810 - val_auc: 0.8802 - val_loss: 0.5904
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5803 - auc: 0.7939 - loss: 0.6556 - val_accuracy: 0.7788 - val_auc: 0.8747 - val_loss: 0.5820
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5949 - auc: 0.8052 - loss: 0.6458 - val_accuracy: 0.8097 - val_auc: 0.8820 - val_loss: 0.5635
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5971 - auc: 0.8108 - loss: 0.6372 - val_accuracy: 0.7677 - val_auc: 0.8698 - val_loss: 0.5999
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [6]:
# ==========================================================
# BLOCK 8 & 9: 3D DENSENET MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Concatenate,
    AveragePooling3D, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for 3D DenseNet.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define 3D DenseNet Building Blocks
# ----------------------------------------------------------
def conv_block(x, growth_rate, name):
    x1 = BatchNormalization(name=name+'_0_bn')(x)
    x1 = Activation('relu', name=name+'_0_relu')(x1)
    x1 = Conv3D(4 * growth_rate, (1, 1, 1), use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_1_conv')(x1)
    
    x1 = BatchNormalization(name=name+'_1_bn')(x1)
    x1 = Activation('relu', name=name+'_1_relu')(x1)
    x1 = Conv3D(growth_rate, (3, 3, 3), padding='same', use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_2_conv')(x1)
    
    x = Concatenate(name=name+'_concat')([x, x1])
    return x

def dense_block(x, blocks, name):
    # Reduced growth rate from 32 to 16 to prevent GPU OOM
    for i in range(blocks):
        x = conv_block(x, growth_rate=16, name=name + '_block' + str(i + 1))
    return x

def transition_block(x, reduction, name):
    x = BatchNormalization(name=name+'_bn')(x)
    x = Activation('relu', name=name+'_relu')(x)
    x = Conv3D(int(tf.keras.backend.int_shape(x)[-1] * reduction), (1, 1, 1), use_bias=False, kernel_regularizer=l2(1e-5), name=name+'_conv')(x)
    x = AveragePooling3D((2, 2, 2), strides=(2, 2, 2), name=name+'_pool')(x)
    return x

def create_densenet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(64, (7, 7, 7), strides=(1, 2, 2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5), name='conv1_conv')(video_input)
    x = BatchNormalization(name='conv1_bn')(x)
    x = Activation('relu', name='conv1_relu')(x)
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same', name='pool1')(x)
    
    # Scaled down depth blocks for 3D Video memory limits
    x = dense_block(x, blocks=4, name='conv2')
    x = transition_block(x, 0.5, name='pool2')
    
    x = dense_block(x, blocks=8, name='conv3')
    x = transition_block(x, 0.5, name='pool3')
    
    x = dense_block(x, blocks=12, name='conv4')
    x = transition_block(x, 0.5, name='pool4')
    
    x = dense_block(x, blocks=8, name='conv5')
    
    x = BatchNormalization(name='bn')(x)
    x = Activation('relu')(x)
    x = GlobalAveragePooling3D(name='global_avg_pool')(x)
    
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid', name='fc_output')(x)

    return Model(inputs=video_input, outputs=output, name='DenseNet3D_Lite')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_densenet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50 
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_densenet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for 3D DenseNet.

Training Model: DenseNet3D_Lite...
Epoch 1/50


2026-04-25 08:22:20.928240: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_fusion_85', 28 bytes spill stores, 28 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 109s 64ms/step - accuracy: 0.5240 - auc: 0.7217 - loss: 0.6812 - val_accuracy: 0.7699 - val_auc: 0.8537 - val_loss: 0.5917
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 60ms/step - accuracy: 0.5502 - auc: 0.7630 - loss: 0.6469 - val_accuracy: 0.6195 - val_auc: 0.8147 - val_loss: 0.8978
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 60ms/step - accuracy: 0.5767 - auc: 0.7806 - loss: 0.6314 - val_accuracy: 0.8009 - val_auc: 0.8700 - val_loss: 0.5538
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 60ms/step - accuracy: 0.5797 - auc: 0.7963 - loss: 0.6215 - val_accuracy: 0.7102 - val_auc: 0.7833 - val_loss: 0.6974
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 60ms/step - accuracy: 0.6018 - auc: 0.8124 - loss: 0.6064 - val_accuracy: 0.7522 - val_auc: 0.8593 - val_loss: 0.6129
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 60ms/step - accuracy: 0.5974 - auc: 0.8132 - loss: 0.6026 - val_accuracy: 0.6283 - val_auc: 0.7941 - val_loss: 0.7090
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 10 & 11: CONVNEXT-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, LayerNormalization, Dense, GlobalAveragePooling3D, 
    Dropout, Activation, Permute, Reshape, Add
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ConvNeXt-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.05 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ConvNeXt-3D Building Blocks
# ----------------------------------------------------------
class ConvNeXtBlock(tf.keras.layers.Layer):
    def __init__(self, dim, drop_path=0., **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.dwconv = Conv3D(dim, kernel_size=7, padding='same', groups=dim)
        self.norm = LayerNormalization(epsilon=1e-6)
        
        self.pwconv1 = Dense(4 * dim) 
        self.act = Activation('gelu')
        
        self.pwconv2 = Dense(dim)
        self.drop_path = Dropout(drop_path) if drop_path > 0. else tf.identity

    def call(self, inputs):
        input_tensor = inputs
        x = self.dwconv(inputs)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = self.drop_path(x)
        return input_tensor + x

def create_convnext3d_model(input_shape, depths=[3, 3, 9, 3], dims=[64, 128, 256, 512]):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(dims[0], kernel_size=(2, 4, 4), strides=(2, 4, 4), padding='valid', name='stem_conv')(video_input)
    x = LayerNormalization(epsilon=1e-6, name='stem_ln')(x)
    
    for i in range(4):
        dim = dims[i]
        depth = depths[i]
        
        for j in range(depth):
            x = ConvNeXtBlock(dim, name=f'stage{i}_block{j}')(x)
            
        if i < 3:
            x = LayerNormalization(epsilon=1e-6, name=f'stage{i}_downsample_ln')(x)
            x = Conv3D(dims[i+1], kernel_size=2, strides=2, padding='valid', name=f'stage{i}_downsample_conv')(x)

    x = GlobalAveragePooling3D()(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='ConvNeXt3D_Nano')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_convnext3d_model(input_shape, depths=[2, 2, 6, 2], dims=[48, 96, 192, 384])

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_convnext3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ConvNeXt-3D.

Training Model: ConvNeXt3D_Nano...
Epoch 1/50


2026-04-25 08:59:06.075757: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 20 bytes spill stores, 20 bytes spill loads

2026-04-25 08:59:06.251378: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 24 bytes spill stores, 24 bytes spill loads

2026-04-25 08:59:06.258630: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 24 bytes spill stores, 24 bytes spill loads

2026-04-25 08:59:06.319271: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot', 408 bytes spill stores, 408 bytes spill loads

2026-04-25 08:59:06.327561: I external/local_xla/xla/strea

  1/906 ━━━━━━━━━━━━━━━━━━━━ 4:39:49 19s/step - accuracy: 0.7500 - auc: 1.0000 - loss: 0.3835

2026-04-25 08:59:16.698126: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_14', 4 bytes spill stores, 4 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_9', 4 bytes spill stores, 4 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 61s 47ms/step - accuracy: 0.3819 - auc: 0.5121 - loss: 0.7940 - val_accuracy: 0.5000 - val_auc: 0.6316 - val_loss: 0.7120
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.3910 - auc: 0.5185 - loss: 0.7155 - val_accuracy: 0.5000 - val_auc: 0.7202 - val_loss: 0.6884
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.3918 - auc: 0.5306 - loss: 0.7057 - val_accuracy: 0.5177 - val_auc: 0.4690 - val_loss: 0.6929
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.3830 - auc: 0.5105 - loss: 0.7054 - val_accuracy: 0.5000 - val_auc: 0.5022 - val_loss: 0.7044
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.3681 - auc: 0.4913 - loss: 0.7073 - val_accuracy: 0.5000 - val_auc: 0.6350 - val_loss: 0.7104
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.3739 - auc: 0.4919 - loss: 0.7032 - val_accuracy: 0.5000 - val_auc: 0.6416 - val_loss: 0.6956
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==========================================================
# BLOCK 12 & 13: RESNEXT-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResNeXt-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ResNeXt-3D Building Blocks
# ----------------------------------------------------------
def grouped_conv3d(x, filters, kernel_size, strides=(1,1,1), padding='same', groups=8):
    try:
        return Conv3D(filters, kernel_size, strides=strides, padding=padding, 
                      groups=groups, use_bias=False, kernel_regularizer=l2(1e-5))(x)
    except:
        group_list = []
        channels_per_group = filters // groups
        splits = tf.split(x, groups, axis=-1)
        for i in range(groups):
            g = Conv3D(channels_per_group, kernel_size, strides=strides, padding=padding, 
                       use_bias=False, kernel_regularizer=l2(1e-5))(splits[i])
            group_list.append(g)
        return Concatenate(axis=-1)(group_list)

def resnext_block(x, filters, strides=(1,1,1), groups=8):
    shortcut = x
    bottleneck_width = filters // 2 

    x = Conv3D(bottleneck_width, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = grouped_conv3d(x, bottleneck_width, (3,3,3), strides=strides, padding='same', groups=groups)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)

    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_resnext3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # Scaled down stem filters from 64 to 32
    x = Conv3D(32, (7,7,7), strides=(1,2,2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(video_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x)

    # Scaled down depths and set groups=8 to fit 16GB VRAM
    x = resnext_block(x, 64, groups=8)
    x = resnext_block(x, 64, groups=8)

    x = resnext_block(x, 128, strides=(2,2,2), groups=8)
    x = resnext_block(x, 128, groups=8)

    x = resnext_block(x, 256, strides=(2,2,2), groups=8)
    x = resnext_block(x, 256, groups=8)

    x = resnext_block(x, 512, strides=(2,2,2), groups=8)
    x = resnext_block(x, 512, groups=8)

    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='ResNeXt3D_Lite')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnext3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resnext3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResNeXt-3D.


I0000 00:00:1777093252.508302 2274037 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13569 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: ResNeXt3D_Lite...
Epoch 1/50


I0000 00:00:1777093257.786107 2274140 service.cc:152] XLA service 0x7fa3c0003960 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777093257.786121 2274140 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-25 11:00:58.026040: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777093259.061904 2274140 cuda_dnn.cc:529] Loaded cuDNN version 91002


  3/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.2500 - auc: 0.0909 - loss: 0.8500       

2026-04-25 11:01:07.994140: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_reduce_select_fusion', 16 bytes spill stores, 16 bytes spill loads

I0000 00:00:1777093268.049201 2274140 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 57s 47ms/step - accuracy: 0.4705 - auc: 0.6520 - loss: 0.7971 - val_accuracy: 0.6770 - val_auc: 0.7854 - val_loss: 0.6346
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.5152 - auc: 0.7118 - loss: 0.7284 - val_accuracy: 0.7323 - val_auc: 0.8478 - val_loss: 0.6002
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5516 - auc: 0.7567 - loss: 0.6869 - val_accuracy: 0.7677 - val_auc: 0.8635 - val_loss: 0.5582
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5615 - auc: 0.7735 - loss: 0.6462 - val_accuracy: 0.8009 - val_auc: 0.8641 - val_loss: 0.5475
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5723 - auc: 0.7929 - loss: 0.6280 - val_accuracy: 0.8053 - val_auc: 0.8833 - val_loss: 0.5242
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5828 - auc: 0.7914 - loss: 0.6173 - val_accuracy: 0.7832 - val_auc: 0.8848 - val_loss: 0.5277
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 14 & 15: EFFICIENTNET-3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Multiply,
    Reshape
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for EfficientNet-3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-5
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define EfficientNet-3D Building Blocks
# ----------------------------------------------------------
def get_activation(activation='swish'):
    return Activation(tf.nn.swish)

def squeeze_excitation_block(x, input_channels, squeeze_ratio=0.25):
    reduced_channels = max(1, int(input_channels * squeeze_ratio))
    
    se = GlobalAveragePooling3D()(x)
    se = Reshape((1, 1, 1, input_channels))(se)
    
    se = Dense(reduced_channels, kernel_initializer='he_normal', use_bias=True)(se)
    se = get_activation('swish')(se)
    
    se = Dense(input_channels, kernel_initializer='he_normal', use_bias=True)(se)
    se = Activation('sigmoid')(se)
    
    x = Multiply()([x, se])
    return x

def mbconv_block(x, input_filters, output_filters, kernel_size, strides, expand_ratio, use_se=True, drop_rate=0.0):
    shortcut = x 
    
    expanded_filters = input_filters * expand_ratio
    if expand_ratio != 1:
        x = Conv3D(expanded_filters, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x)
        x = get_activation('swish')(x)
    
    # Factorized (2+1)D approach to avoid 3D Dense Kernel explosion
    # 1. Spatial Convolution
    x = Conv3D(expanded_filters, (1, kernel_size, kernel_size), strides=(1, strides[1], strides[2]), padding='same', 
               use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    # 2. Temporal Convolution
    x = Conv3D(expanded_filters, (kernel_size, 1, 1), strides=(strides[0], 1, 1), padding='same', 
               use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    if use_se:
        x = squeeze_excitation_block(x, expanded_filters)
    
    x = Conv3D(output_filters, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    
    if strides == (1, 1, 1) and input_filters == output_filters:
        if drop_rate > 0:
            x = Dropout(drop_rate)(x)
        x = Add()([shortcut, x]) 
    
    return x

def create_efficientnet3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # Stem
    x = Conv3D(16, 3, strides=(2, 2, 2), padding='same', use_bias=False, kernel_initializer='he_normal')(video_input)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    # Stage 1
    x = mbconv_block(x, 16, 8, kernel_size=3, strides=(1,1,1), expand_ratio=1)
    
    # Stage 2
    x = mbconv_block(x, 8, 16, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    
    # Stage 3
    x = mbconv_block(x, 16, 24, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    
    # Stage 4
    x = mbconv_block(x, 24, 48, kernel_size=3, strides=(1,2,2), expand_ratio=2)
    x = mbconv_block(x, 48, 64, kernel_size=3, strides=(1,1,1), expand_ratio=2)
    
    # Stage 5
    x = mbconv_block(x, 64, 96, kernel_size=3, strides=(2,2,2), expand_ratio=2)
    x = mbconv_block(x, 96, 128, kernel_size=3, strides=(1,1,1), expand_ratio=2)
    
    # Head
    x = Conv3D(512, 1, padding='same', use_bias=False, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = get_activation('swish')(x)
    
    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.2)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='EfficientNet3D_Factorized')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_efficientnet3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_efficientnet3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for EfficientNet-3D.

Training Model: EfficientNet3D_Factorized...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 61s 48ms/step - accuracy: 0.5141 - auc: 0.7147 - loss: 0.6454 - val_accuracy: 0.6549 - val_auc: 0.7898 - val_loss: 0.7340
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5502 - auc: 0.7568 - loss: 0.6168 - val_accuracy: 0.6903 - val_auc: 0.8057 - val_loss: 0.6989
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5513 - auc: 0.7691 - loss: 0.6064 - val_accuracy: 0.7544 - val_auc: 0.8276 - val_loss: 0.5995
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5679 - auc: 0.7855 - loss: 0.5959 - val_accuracy: 0.7566 - val_auc: 0.8227 - val_loss: 0.6254
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.5480 - auc: 0.7815 - loss: 0.5900 - val_accuracy: 0.6394 - val_auc: 0.8043 - val_loss: 0.7168
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5717 - auc: 0.7876 - loss: 0.5883 - 

In [6]:
# ==========================================================
# BLOCK 16 & 17: VIVIT MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ViViT.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.01 
LABEL_SMOOTHING = 0.1 
PROJECTION_DIM = 64  
NUM_HEADS = 4
TRANSFORMER_LAYERS = 4

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ViViT Components
# ----------------------------------------------------------
class TubeletEmbedding(layers.Layer):
    def __init__(self, embed_dim, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.projection = layers.Conv3D(
            filters=embed_dim, kernel_size=patch_size,
            strides=patch_size, padding="VALID", name="tubelet_proj"
        )
        self.flatten = layers.Reshape((-1, embed_dim))

    def call(self, videos):
        projected_patches = self.projection(videos)
        flattened_patches = self.flatten(projected_patches)
        return flattened_patches

class PositionalEncoder(layers.Layer):
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    def build(self, input_shape):
        _, num_tokens, _ = input_shape
        self.position_embedding = self.add_weight(
            name="pos_embedding", shape=(num_tokens, self.embed_dim),
            initializer="glorot_uniform", trainable=True
        )

    def call(self, encoded_tokens):
        return encoded_tokens + self.position_embedding

def transformer_encoder_block(inputs, embed_dim, num_heads, ff_dim, dropout=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=embed_dim, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Add()([x, inputs]) 

    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(ff_dim, activation=tf.nn.gelu)(y) 
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(embed_dim)(y)
    y = layers.Add()([y, x]) 
    return y

def create_vivit_model(input_shape):
    video_input = layers.Input(shape=input_shape)

    patches = TubeletEmbedding(embed_dim=PROJECTION_DIM, patch_size=(2, 16, 16))(video_input)
    encoded_patches = PositionalEncoder(embed_dim=PROJECTION_DIM)(patches)

    for _ in range(TRANSFORMER_LAYERS):
        encoded_patches = transformer_encoder_block(
            encoded_patches, embed_dim=PROJECTION_DIM, 
            num_heads=NUM_HEADS, ff_dim=PROJECTION_DIM * 2, dropout=0.1
        )

    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.GlobalAveragePooling1D()(representation)
    
    x = layers.Dropout(0.5)(representation)
    x = layers.Dense(128, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    return Model(inputs=video_input, outputs=output, name="ViViT_Transformer")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_vivit_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_vivit_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ViViT.

Training Model: ViViT_Transformer...
Epoch 1/50


2026-04-25 12:01:11.915281: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_103', 188 bytes spill stores, 188 bytes spill loads

2026-04-25 12:01:11.947389: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_103', 92 bytes spill stores, 92 bytes spill loads

2026-04-25 12:01:12.016759: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_114', 12 bytes spill stores, 12 bytes spill loads

2026-04-25 12:01:12.034612: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_103', 536 bytes spill stores, 536 bytes spill loads

2026-04-25 12:01:12.070302: I external/l

905/906 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.3765 - auc: 0.5104 - loss: 0.7983

2026-04-25 12:01:57.794099: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1286', 8 bytes spill stores, 8 bytes spill loads

2026-04-25 12:01:57.841657: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 568 bytes spill stores, 472 bytes spill loads

2026-04-25 12:01:57.985296: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 128 bytes spill stores, 128 bytes spill loads

2026-04-25 12:01:58.043092: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 192 bytes spill stores, 192 bytes spill loads

2026-04-25 12:01:58.088507: I external/loc

906/906 ━━━━━━━━━━━━━━━━━━━━ 55s 47ms/step - accuracy: 0.3769 - auc: 0.5166 - loss: 0.7666 - val_accuracy: 0.4978 - val_auc: 0.6089 - val_loss: 0.6963
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.4087 - auc: 0.5354 - loss: 0.7187 - val_accuracy: 0.5000 - val_auc: 0.6276 - val_loss: 0.6984
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.3998 - auc: 0.5428 - loss: 0.7111 - val_accuracy: 0.5553 - val_auc: 0.6396 - val_loss: 0.6863
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.3899 - auc: 0.5432 - loss: 0.7039 - val_accuracy: 0.5354 - val_auc: 0.6048 - val_loss: 0.6888
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.4161 - auc: 0.5565 - loss: 0.7007 - val_accuracy: 0.5774 - val_auc: 0.6355 - val_loss: 0.6787
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 39s 43ms/step - accuracy: 0.4106 - auc: 0.5573 - loss: 0.6957 - val_accuracy: 0.5819 - val_auc: 0.6167 - val_loss: 0.6779
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 18 & 19: SLOWFAST MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Concatenate,
    MaxPooling3D, AveragePooling3D, Lambda
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for SlowFast.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define SlowFast Building Blocks
# ----------------------------------------------------------
def slowfast_block(x_slow, x_fast, filters, strides=(1,1,1)):
    # --- SLOW PATH (Spatial focus, 1x3x3) ---
    ys = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_slow)
    ys = BatchNormalization()(ys)
    ys = Activation('relu')(ys)
    
    ys = Conv3D(filters, (1,3,3), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(ys)
    ys = BatchNormalization()(ys)
    ys = Activation('relu')(ys)
    
    ys = Conv3D(filters*4, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(ys)
    ys = BatchNormalization()(ys)

    if strides != (1,1,1) or x_slow.shape[-1] != filters*4:
        shortcut_s = Conv3D(filters*4, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_slow)
        shortcut_s = BatchNormalization()(shortcut_s)
    else:
        shortcut_s = x_slow
        
    # --- FAST PATH (Factorized Spatiotemporal Focus) ---
    fast_filters = max(1, filters // 4)  # Kept ratio tighter for Micro scale
    
    yf = Conv3D(fast_filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_fast)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    # Factorized (3,3,3) into (1,3,3) -> (3,1,1) to prevent XLA OOM crash
    yf = Conv3D(fast_filters, (1,3,3), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    yf = Conv3D(fast_filters, (3,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    yf = Activation('relu')(yf)
    
    yf = Conv3D(fast_filters*4, (1,1,1), strides=(1,1,1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(yf)
    yf = BatchNormalization()(yf)
    
    if strides != (1,1,1) or x_fast.shape[-1] != fast_filters*4:
        shortcut_f = Conv3D(fast_filters*4, (1,1,1), strides=strides, padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x_fast)
        shortcut_f = BatchNormalization()(shortcut_f)
    else:
        shortcut_f = x_fast

    # --- LATERAL CONNECTION ---
    ys = Add()([ys, shortcut_s])
    ys = Activation('relu')(ys)
    
    yf = Add()([yf, shortcut_f])
    yf = Activation('relu')(yf)
    
    return ys, yf

def create_slowfast_model(input_shape):
    video_input = Input(shape=input_shape)
    
    # --- Input Splitting ---
    x_fast = video_input
    x_slow = Lambda(lambda x: x[:, ::4, :, :, :], name='slow_slice')(video_input)
    
    # --- Stem ---
    # Slow Stem
    x_slow = Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(x_slow)
    x_slow = BatchNormalization()(x_slow)
    x_slow = Activation('relu')(x_slow)
    x_slow = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x_slow)
    
    # Fast Stem (Factorized)
    x_fast = Conv3D(4, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(x_fast)
    x_fast = BatchNormalization()(x_fast)
    x_fast = Activation('relu')(x_fast)
    x_fast = Conv3D(4, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x_fast)
    x_fast = BatchNormalization()(x_fast)
    x_fast = Activation('relu')(x_fast)
    x_fast = MaxPooling3D((1,3,3), strides=(1,2,2), padding='same')(x_fast)
    
    # --- Stages ---
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 16)
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 32, strides=(1,2,2))
    x_slow, x_fast = slowfast_block(x_slow, x_fast, 64, strides=(1,2,2))
    
    # --- Fusion & Head ---
    pool_slow = GlobalAveragePooling3D()(x_slow)
    pool_fast = GlobalAveragePooling3D()(x_fast)
    
    x = Concatenate()([pool_slow, pool_fast])
    
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='SlowFast_Micro')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_slowfast_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_slowfast_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for SlowFast.

Training Model: SlowFast_Micro...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 47ms/step - accuracy: 0.4848 - auc: 0.6716 - loss: 0.6811 - val_accuracy: 0.7367 - val_auc: 0.8337 - val_loss: 0.5657
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5621 - auc: 0.7635 - loss: 0.6180 - val_accuracy: 0.7478 - val_auc: 0.8454 - val_loss: 0.5509
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.5533 - auc: 0.7625 - loss: 0.6183 - val_accuracy: 0.7854 - val_auc: 0.8710 - val_loss: 0.5165
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5599 - auc: 0.7750 - loss: 0.6041 - val_accuracy: 0.7699 - val_auc: 0.8705 - val_loss: 0.5249
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5728 - auc: 0.7973 - loss: 0.5897 - val_accuracy: 0.7898 - val_auc: 0.8826 - val_loss: 0.5028
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5698 - auc: 0.7895 - loss: 0.5876 - val_accuracy: 0.78

In [8]:
# ==========================================================
# BLOCK 20 & 21: R(2+1)D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for R(2+1)D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define R(2+1)D Building Blocks
# ----------------------------------------------------------
def conv2plus1d(x, filters, strides=(1,1,1)):
    inter_filters = filters 

    spatial_strides = (1, strides[1], strides[2])
    x = Conv3D(inter_filters, (1, 3, 3), strides=spatial_strides, padding='same', 
               use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    temporal_strides = (strides[0], 1, 1)
    x = Conv3D(filters, (3, 1, 1), strides=temporal_strides, padding='same', 
               use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    return x

def r2plus1d_block(x, filters, strides=(1,1,1)):
    shortcut = x
    
    x = conv2plus1d(x, filters, strides=strides)
    x = conv2plus1d(x, filters)
    
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', 
                          use_bias=False, kernel_regularizer=l2(1e-5))(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def create_r2plus1d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(45, (1, 7, 7), strides=(1, 2, 2), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(video_input)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv3D(64, (3, 1, 1), strides=(1, 1, 1), padding='same', use_bias=False, kernel_regularizer=l2(1e-5))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = r2plus1d_block(x, 64)
    x = r2plus1d_block(x, 64)
    
    x = r2plus1d_block(x, 128, strides=(2, 2, 2))
    x = r2plus1d_block(x, 128)
    
    x = r2plus1d_block(x, 256, strides=(2, 2, 2))
    x = r2plus1d_block(x, 256)
    
    x = r2plus1d_block(x, 512, strides=(2, 2, 2))
    x = r2plus1d_block(x, 512)

    x = GlobalAveragePooling3D()(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='R2Plus1D_18')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_r2plus1d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_r2plus1d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for R(2+1)D.

Training Model: R2Plus1D_18...
Epoch 1/50


2026-04-25 13:09:35.822200: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.


906/906 ━━━━━━━━━━━━━━━━━━━━ 85s 76ms/step - accuracy: 0.4925 - auc: 0.6799 - loss: 0.8399 - val_accuracy: 0.7323 - val_auc: 0.8006 - val_loss: 0.7210
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 67s 74ms/step - accuracy: 0.5607 - auc: 0.7582 - loss: 0.7129 - val_accuracy: 0.7655 - val_auc: 0.8461 - val_loss: 0.6346
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 67s 74ms/step - accuracy: 0.5643 - auc: 0.7761 - loss: 0.6870 - val_accuracy: 0.6327 - val_auc: 0.6398 - val_loss: 1.8119
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 67s 74ms/step - accuracy: 0.5927 - auc: 0.7921 - loss: 0.6611 - val_accuracy: 0.8053 - val_auc: 0.8871 - val_loss: 0.6060
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 67s 74ms/step - accuracy: 0.5866 - auc: 0.7979 - loss: 0.6540 - val_accuracy: 0.7456 - val_auc: 0.8725 - val_loss: 0.6460
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 67s 73ms/step - accuracy: 0.5982 - auc: 0.8186 - loss: 0.6367 - val_accuracy: 0.7522 - val_auc: 0.8833 - val_loss: 0.6347
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==========================================================
# BLOCK 22 & 23: X3D MODEL TRAINING & THESIS EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, Add, Multiply, Reshape
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for X3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 5e-5  # X3D is lightweight, needs less decay
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define X3D Building Blocks
# ----------------------------------------------------------
def swish(x):
    return Activation(tf.nn.swish)(x)

def se_block(x, filters, ratio=0.25):
    inputs = x
    x = GlobalAveragePooling3D()(x)
    x = Reshape((1, 1, 1, filters))(x)
    
    reduced_filters = max(1, int(filters * ratio))
    x = Dense(reduced_filters, kernel_initializer='he_normal', use_bias=True)(x)
    x = swish(x)
    x = Dense(filters, kernel_initializer='he_normal', use_bias=True)(x)
    x = Activation('sigmoid')(x)
    
    return Multiply()([inputs, x])

def x3d_bottleneck(x, filters, strides=(1,1,1), expansion_ratio=2.25):
    shortcut = x
    input_filters = x.shape[-1]
    expanded_filters = int(input_filters * expansion_ratio)

    x = Conv3D(expanded_filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = Conv3D(expanded_filters, (3,3,3), strides=strides, padding='same', 
               groups=expanded_filters, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = se_block(x, expanded_filters)

    x = Conv3D(filters, (1,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)

    if strides != (1,1,1) or input_filters != filters:
        shortcut = Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    x = Add()([x, shortcut])
    return x 

def create_x3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = Conv3D(24, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = BatchNormalization()(x)
    x = swish(x)
    
    x = Conv3D(24, (5,1,1), strides=(1,1,1), padding='same', groups=24, use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)

    x = x3d_bottleneck(x, 24, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 24, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 48, strides=(1,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 48, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 48, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 96, strides=(2,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    x = x3d_bottleneck(x, 96, expansion_ratio=2.25)
    
    x = x3d_bottleneck(x, 192, strides=(1,2,2), expansion_ratio=2.25)
    x = x3d_bottleneck(x, 192, expansion_ratio=2.25)
    
    x = Conv3D(432, (1,1,1), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = swish(x)
    
    x = GlobalAveragePooling3D()(x)
    
    x = Dense(2048, activation='relu')(x) 
    x = Dropout(0.5)(x)
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='X3D_M')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_x3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_x3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for X3D.


I0000 00:00:1777127407.593932 2323658 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13600 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: X3D_M...
Epoch 1/50


I0000 00:00:1777127417.388191 2323765 service.cc:152] XLA service 0x76c258003910 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777127417.388210 2323765 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-25 20:30:17.873464: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777127419.860208 2323765 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-25 20:30:21.260964: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 108 bytes spill stores, 108 bytes spill loads

2026-04-25 20:30:21.426685: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_

  1/906 ━━━━━━━━━━━━━━━━━━━━ 7:13:35 29s/step - accuracy: 0.5000 - auc: 0.0000e+00 - loss: 0.6679

I0000 00:00:1777127436.980495 2323765 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.5031 - auc: 0.6787 - loss: 0.6669

2026-04-25 20:32:17.012177: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 28 bytes spill stores, 28 bytes spill loads

2026-04-25 20:32:17.054274: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22', 28 bytes spill stores, 28 bytes spill loads

2026-04-25 20:32:17.190803: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 124 bytes spill stores, 124 bytes spill loads

2026-04-25 20:32:17.201832: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_12', 124 bytes spill stores, 124 bytes spill loads

2026-04-25 20:32:17.410200: I external/local

906/906 ━━━━━━━━━━━━━━━━━━━━ 134s 117ms/step - accuracy: 0.5301 - auc: 0.7232 - loss: 0.6318 - val_accuracy: 0.7611 - val_auc: 0.8649 - val_loss: 0.5361
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 101s 112ms/step - accuracy: 0.5797 - auc: 0.7912 - loss: 0.5845 - val_accuracy: 0.6217 - val_auc: 0.7918 - val_loss: 0.6853
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 102s 113ms/step - accuracy: 0.5905 - auc: 0.8017 - loss: 0.5715 - val_accuracy: 0.8053 - val_auc: 0.8672 - val_loss: 0.5013
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 102s 113ms/step - accuracy: 0.5977 - auc: 0.8117 - loss: 0.5565 - val_accuracy: 0.8186 - val_auc: 0.8968 - val_loss: 0.4771
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 101s 112ms/step - accuracy: 0.6115 - auc: 0.8293 - loss: 0.5416 - val_accuracy: 0.7942 - val_auc: 0.8801 - val_loss: 0.4927
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 102s 112ms/step - accuracy: 0.6305 - auc: 0.8409 - loss: 0.5286 - val_accuracy: 0.8473 - val_auc: 0.9161 - val_loss: 0.4456
Epoch 7/50
906/906 ━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 24 & 25: C3D (MODERNIZED) MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv3D, BatchNormalization, Activation, 
    GlobalAveragePooling3D, Dense, Dropout, MaxPooling3D
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for C3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define C3D Building Blocks
# ----------------------------------------------------------
def c3d_block(x, filters, count):
    for _ in range(count):
        x = Conv3D(filters, (3, 3, 3), activation='relu', padding='same', 
                   use_bias=False, kernel_regularizer=l2(1e-5))(x)
        x = BatchNormalization()(x)
    return x

def create_c3d_model(input_shape):
    video_input = Input(shape=input_shape)
    
    x = c3d_block(video_input, 64, 1)
    x = MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 128, 1)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 256, 2)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 512, 2)
    x = MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2), padding='same')(x)
    
    x = c3d_block(x, 512, 2)
    
    x = GlobalAveragePooling3D()(x)
    
    x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    x = Dense(512, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.5)(x)
    
    output = Dense(1, activation='sigmoid')(x)

    return Model(inputs=video_input, outputs=output, name='C3D_Modernized')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_c3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_c3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for C3D.

Training Model: C3D_Modernized...
Epoch 1/50


2026-04-25 21:55:23.551443: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_2695', 96 bytes spill stores, 96 bytes spill loads

2026-04-25 21:55:24.514968: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-25 21:55:24.607562: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-25 21:55:34.506168: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:382] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are r

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.4908 - auc: 0.6765 - loss: 0.8026

2026-04-25 21:58:17.061541: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_327', 64 bytes spill stores, 64 bytes spill loads

2026-04-25 21:58:17.444247: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-25 21:58:17.537275: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


906/906 ━━━━━━━━━━━━━━━━━━━━ 185s 186ms/step - accuracy: 0.5041 - auc: 0.6926 - loss: 0.7964 - val_accuracy: 0.7566 - val_auc: 0.8431 - val_loss: 0.6725
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 163s 180ms/step - accuracy: 0.5381 - auc: 0.7431 - loss: 0.7560 - val_accuracy: 0.7588 - val_auc: 0.8338 - val_loss: 0.7839
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 163s 180ms/step - accuracy: 0.5317 - auc: 0.7423 - loss: 0.7513 - val_accuracy: 0.7920 - val_auc: 0.8559 - val_loss: 0.6488
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 164s 181ms/step - accuracy: 0.5607 - auc: 0.7663 - loss: 0.7325 - val_accuracy: 0.7500 - val_auc: 0.8718 - val_loss: 0.6500
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 163s 180ms/step - accuracy: 0.5825 - auc: 0.7883 - loss: 0.7160 - val_accuracy: 0.7832 - val_auc: 0.8557 - val_loss: 0.6506
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 163s 180ms/step - accuracy: 0.5786 - auc: 0.7933 - loss: 0.7008 - val_accuracy: 0.8142 - val_auc: 0.8672 - val_loss: 0.6223
Epoch 7/50
906/906 ━━━━━━━━

In [6]:
# ==========================================================
# BLOCK 26 & 27: CNN-TRANSFORMER HYBRID TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for CNN-Transformer Hybrid.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 0.01  # Transformers need higher decay
LABEL_SMOOTHING = 0.1
EMBED_DIM = 128      # Dimension for Transformer
NUM_HEADS = 4
TRANSFORMER_LAYERS = 2

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Model Components
# ----------------------------------------------------------
def create_cnn_feature_extractor(input_shape):
    cnn_input = layers.Input(shape=input_shape)
    
    # Stem
    x = layers.Conv2D(32, (7, 7), strides=2, padding='same', use_bias=False)(cnn_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding='same')(x)
    
    # Residual Block 1
    shortcut = x
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if shortcut.shape[-1] != 64:
        shortcut = layers.Conv2D(64, (1, 1), padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    # Residual Block 2 (Downsample)
    shortcut = x
    x = layers.Conv2D(128, (3, 3), strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    shortcut = layers.Conv2D(128, (1, 1), strides=2, padding='same', use_bias=False)(shortcut)
    shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    
    # Spatial Aggregation
    x = layers.GlobalAveragePooling2D()(x)
    
    return models.Model(inputs=cnn_input, outputs=x, name="cnn_extractor")

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.output_dim = output_dim

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_positions = self.position_embeddings(positions)
        return inputs + embedded_positions

def create_cnn_transformer_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- 1. Spatial Feature Extraction (CNN) ---
    cnn_extractor = create_cnn_feature_extractor(input_shape[1:]) 
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input) 
    
    # --- 2. Temporal Modeling (Transformer) ---
    x = layers.Dense(EMBED_DIM)(encoded_frames)
    x = PositionalEmbedding(sequence_length=NUM_FRAMES, output_dim=EMBED_DIM)(x)
    
    for _ in range(TRANSFORMER_LAYERS):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS, key_dim=EMBED_DIM, dropout=0.1
        )(x1, x1)
        x2 = layers.Add()([attention_output, x]) 

        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = layers.Dense(EMBED_DIM * 2, activation=tf.nn.gelu)(x3)
        x3 = layers.Dropout(0.1)(x3)
        x3 = layers.Dense(EMBED_DIM)(x3)
        x = layers.Add()([x3, x2]) 

    # --- 3. Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation="sigmoid")(x)

    return models.Model(inputs=video_input, outputs=output, name="CNN_Transformer_Hybrid")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_cnn_transformer_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_cnn_transformer_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024)

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for CNN-Transformer Hybrid.

Training Model: CNN_Transformer_Hybrid...
Epoch 1/50


2026-04-26 00:12:23.372260: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17', 192 bytes spill stores, 192 bytes spill loads

2026-04-26 00:12:23.422347: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17_0', 40 bytes spill stores, 48 bytes spill loads

2026-04-26 00:12:23.673717: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_17_0', 184 bytes spill stores, 184 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 63s 47ms/step - accuracy: 0.4329 - auc: 0.5870 - loss: 0.7468 - val_accuracy: 0.6770 - val_auc: 0.7830 - val_loss: 0.6230
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.5069 - auc: 0.6948 - loss: 0.6709 - val_accuracy: 0.7655 - val_auc: 0.8288 - val_loss: 0.5745
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5279 - auc: 0.7244 - loss: 0.6423 - val_accuracy: 0.7456 - val_auc: 0.8430 - val_loss: 0.5891
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 40s 44ms/step - accuracy: 0.5475 - auc: 0.7506 - loss: 0.6323 - val_accuracy: 0.7810 - val_auc: 0.8562 - val_loss: 0.5878
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5494 - auc: 0.7632 - loss: 0.6232 - val_accuracy: 0.7743 - val_auc: 0.8578 - val_loss: 0.5520
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 38s 42ms/step - accuracy: 0.5552 - auc: 0.7710 - loss: 0.6114 - val_accuracy: 0.7699 - val_auc: 0.8672 - val_loss: 0.5288
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================================
# BLOCK 28 & 29: CONVLSTM HYBRID MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ConvLSTM.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ConvLSTM Components
# ----------------------------------------------------------
def create_cnn_backbone(input_shape):
    inputs = layers.Input(shape=input_shape)
    
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    x = layers.Conv2D(512, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x) 
    
    return models.Model(inputs=inputs, outputs=x, name="cnn_backbone")

def create_convlstm_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    cnn = create_cnn_backbone(input_shape[1:])
    x = layers.TimeDistributed(cnn)(video_input)
    
    x = layers.ConvLSTM2D(
        filters=64, 
        kernel_size=(3, 3), 
        padding='same', 
        return_sequences=False, 
        dropout=0.2,
        recurrent_dropout=0.0 
    )(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="ConvLSTM_Hybrid")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_convlstm_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_convlstm_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ConvLSTM.

Training Model: ConvLSTM_Hybrid...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.4194 - auc: 0.5702 - loss: 0.7011

2026-04-26 00:47:18.516885: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3396', 8 bytes spill stores, 8 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 69s 61ms/step - accuracy: 0.4663 - auc: 0.6533 - loss: 0.6729 - val_accuracy: 0.7478 - val_auc: 0.8403 - val_loss: 0.5683
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 58ms/step - accuracy: 0.5621 - auc: 0.7745 - loss: 0.6105 - val_accuracy: 0.7699 - val_auc: 0.8689 - val_loss: 0.5256
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 59ms/step - accuracy: 0.5657 - auc: 0.7779 - loss: 0.5943 - val_accuracy: 0.7965 - val_auc: 0.8887 - val_loss: 0.5099
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 58ms/step - accuracy: 0.5971 - auc: 0.8106 - loss: 0.5761 - val_accuracy: 0.7832 - val_auc: 0.8853 - val_loss: 0.5170
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 58ms/step - accuracy: 0.5764 - auc: 0.8070 - loss: 0.5788 - val_accuracy: 0.8208 - val_auc: 0.8899 - val_loss: 0.5025
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 53s 58ms/step - accuracy: 0.5982 - auc: 0.8314 - loss: 0.5643 - val_accuracy: 0.7721 - val_auc: 0.8932 - val_loss: 0.5231
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [8]:
# ==========================================================
# BLOCK 30 & 31: TRANSFER LEARNING (MOBILENETV2 + LSTM) & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Transfer Learning.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-5 
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Transfer Learning Model
# ----------------------------------------------------------
def create_transfer_mobilenet_lstm(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- The Backbone (ImageNet Pre-trained) ---
    base_cnn = MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # FREEZE the backbone (Critical for Transfer Learning)
    base_cnn.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='mobilenet_feature_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input)
    
    # --- Temporal Modeling (LSTM) ---
    x = layers.LSTM(256, return_sequences=False, dropout=0.3)(encoded_frames)
    
    # --- Classification Head ---
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="TL_MobileNet_LSTM")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_transfer_mobilenet_lstm(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_tl_mobilenet_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Transfer Learning.

Training Model: TL_MobileNet_LSTM...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 71s 58ms/step - accuracy: 0.5751 - auc: 0.7755 - loss: 0.6080 - val_accuracy: 0.7788 - val_auc: 0.9032 - val_loss: 0.5193
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.6156 - auc: 0.8400 - loss: 0.5478 - val_accuracy: 0.8009 - val_auc: 0.9097 - val_loss: 0.4981
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 51ms/step - accuracy: 0.6258 - auc: 0.8478 - loss: 0.5364 - val_accuracy: 0.8274 - val_auc: 0.9012 - val_loss: 0.4789
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.6308 - auc: 0.8468 - loss: 0.5279 - val_accuracy: 0.8230 - val_auc: 0.9161 - val_loss: 0.4681
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.6162 - auc: 0.8526 - loss: 0.5285 - val_accuracy: 0.8252 - val_auc: 0.9156 - val_loss: 0.4555
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 51ms/step - accuracy: 0.6363 - auc: 0.8587 - loss: 0.5139 - val_ac

In [4]:
# ==========================================================
# BLOCK 32 & 33: TRANSFER LEARNING (RESNET50 + ATTENTION) & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResNet50 Transfer.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4 
LABEL_SMOOTHING = 0.1 
ATTENTION_HEADS = 4

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Transfer Learning Model
# ----------------------------------------------------------
def create_resnet_attention_model(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- The Backbone (ResNet50 - Heavyweight) ---
    base_cnn = ResNet50(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # FREEZE the backbone
    base_cnn.trainable = False
    
    # Pooling immediately to save memory (2048 features per frame)
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='resnet_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input) 
    
    # --- Temporal Modeling (Attention) ---
    x = layers.LayerNormalization(epsilon=1e-6)(encoded_frames)
    
    attention_output = layers.MultiHeadAttention(
        num_heads=ATTENTION_HEADS, 
        key_dim=2048 // ATTENTION_HEADS, 
        dropout=0.1
    )(x, x)
    
    x = layers.Add()([x, attention_output])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    # --- Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="TL_ResNet50_Attention")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resnet_attention_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_tl_resnet_attn_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResNet50 Transfer.


I0000 00:00:1781026472.879067 1684813 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13011 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: TL_ResNet50_Attention...
Epoch 1/50


I0000 00:00:1781026498.672875 1685131 service.cc:152] XLA service 0x7e88f88282d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1781026498.672891 1685131 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-06-09 23:35:00.310995: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1781026509.714471 1685131 cuda_dnn.cc:529] Loaded cuDNN version 91900
2026-06-09 23:35:11.593386: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_19', 232 bytes spill stores, 232 bytes spill loads

2026-06-09 23:35:11.617437: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_

  1/906 ━━━━━━━━━━━━━━━━━━━━ 10:46:07 43s/step - accuracy: 0.2500 - auc: 0.6667 - loss: 0.9290

I0000 00:00:1781026516.816175 1685131 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.3738 - auc: 0.5047 - loss: 0.9705

2026-06-09 23:36:22.091176: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 12 bytes spill stores, 12 bytes spill loads

2026-06-09 23:36:22.304754: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 104 bytes spill stores, 104 bytes spill loads

2026-06-09 23:36:22.549800: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 4024 bytes spill stores, 4004 bytes spill loads

2026-06-09 23:36:22.564398: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22312', 3876 bytes spill stores, 3868 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 118s 83ms/step - accuracy: 0.3711 - auc: 0.4987 - loss: 0.7976 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7242
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 59s 65ms/step - accuracy: 0.3849 - auc: 0.5099 - loss: 0.7221 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7168
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 59s 65ms/step - accuracy: 0.3750 - auc: 0.4973 - loss: 0.7224 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7122
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 59s 65ms/step - accuracy: 0.3656 - auc: 0.4908 - loss: 0.7167 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7084
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 59s 65ms/step - accuracy: 0.3709 - auc: 0.4956 - loss: 0.7120 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7052
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 59s 65ms/step - accuracy: 0.3601 - auc: 0.4860 - loss: 0.7077 - val_accuracy: 0.5000 - val_auc: 0.5000 - val_loss: 0.7023
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━

In [10]:
# ==========================================================
# BLOCK 34 & 35: FINE-TUNED DENSENET121 + BiLSTM & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Fine-Tuning.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
FINE_TUNE_LR = 1e-5 
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking (Thesis Requirement)
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Fine-Tuning Model
# ----------------------------------------------------------
def create_finetuned_densenet_bilstm(input_shape):
    video_input = layers.Input(shape=input_shape) 
    
    # --- Backbone: DenseNet121 (ImageNet) ---
    base_cnn = DenseNet121(
        weights='imagenet', 
        include_top=False, 
        input_shape=input_shape[1:] 
    )
    
    # --- FINE-TUNING STRATEGY ---
    base_cnn.trainable = False
    
    # Unfreeze the last 50 layers
    for layer in base_cnn.layers[-50:]: 
        layer.trainable = True
        
    x = layers.GlobalAveragePooling2D()(base_cnn.output)
    cnn_extractor = models.Model(inputs=base_cnn.input, outputs=x, name='densenet_extractor')
    
    # --- Time Distributed Application ---
    encoded_frames = layers.TimeDistributed(cnn_extractor)(video_input)
    
    # --- Temporal Modeling: Bidirectional LSTM ---
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=False, dropout=0.3))(encoded_frames)
    
    # --- Classification Head ---
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name="DenseNet_BiLSTM_FineTuned")

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_finetuned_densenet_bilstm(input_shape)

optimizer = optimizers.AdamW(learning_rate=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_finetuned_densenet_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Fine-Tuning.

Training Model: DenseNet_BiLSTM_FineTuned...
Epoch 1/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 259s 210ms/step - accuracy: 0.4398 - auc: 0.6017 - loss: 0.7097 - val_accuracy: 0.7434 - val_auc: 0.8474 - val_loss: 0.6014
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 170s 187ms/step - accuracy: 0.5212 - auc: 0.7172 - loss: 0.6577 - val_accuracy: 0.8142 - val_auc: 0.8849 - val_loss: 0.5342
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 170s 187ms/step - accuracy: 0.5491 - auc: 0.7514 - loss: 0.6354 - val_accuracy: 0.8186 - val_auc: 0.9010 - val_loss: 0.4986
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 170s 188ms/step - accuracy: 0.5563 - auc: 0.7888 - loss: 0.6174 - val_accuracy: 0.8119 - val_auc: 0.9138 - val_loss: 0.4802
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 170s 187ms/step - accuracy: 0.5690 - auc: 0.7924 - loss: 0.6058 - val_accuracy: 0.8053 - val_auc: 0.9200 - val_loss: 0.4683
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 170s 188ms/step - accuracy: 0.5748 - auc: 0.8008 - loss: 0

In [4]:
# ==========================================================
# BLOCK 36 & 37: NANO3D MODEL TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation for attention-driven channel weighting."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_block(x, filters, strides=(1,1,1)):
    """Factorized Spatiotemporal Block with SE & Residual Connection."""
    shortcut = x
    
    # Spatial feature extraction
    x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Temporal feature extraction
    x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Stem ---
    x = layers.Conv3D(16, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Conv3D(16, (3,1,1), strides=(1,1,1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Hierarchical Blocks ---
    # Block 1 (Low-level features)
    b1 = nano_block(x, 32)
    b1 = nano_block(b1, 32)
    
    # Block 2 (Mid-level features)
    b2 = nano_block(b1, 64, strides=(1,2,2))
    b2 = nano_block(b2, 64)
    
    # Block 3 (High-level features)
    b3 = nano_block(b2, 128, strides=(2,2,2))
    b3 = nano_block(b3, 128)
    
    # --- Multi-Scale Feature Fusion (DenseNet replacement) ---
    pool1 = layers.GlobalAveragePooling3D()(b1)
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    
    merged_features = layers.Concatenate()([pool1, pool2, pool3])
    
    # --- Classification Head ---
    x = layers.Dropout(0.4)(merged_features)
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.Dropout(0.4)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D.


I0000 00:00:1777208027.654464 3803872 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13599 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: Nano3D...
Epoch 1/50


I0000 00:00:1777208032.152930 3804019 service.cc:152] XLA service 0x77590003b2b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777208032.152945 3804019 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-04-26 18:53:52.334754: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777208033.189662 3804019 cuda_dnn.cc:529] Loaded cuDNN version 91002


  3/906 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.4583 - auc: 0.4815 - loss: 0.7408   

I0000 00:00:1777208040.258108 3804019 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 60s 53ms/step - accuracy: 0.5428 - auc: 0.7472 - loss: 0.6425 - val_accuracy: 0.7854 - val_auc: 0.8684 - val_loss: 0.5382
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 49ms/step - accuracy: 0.5668 - auc: 0.7791 - loss: 0.6134 - val_accuracy: 0.7920 - val_auc: 0.8839 - val_loss: 0.5292
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 49ms/step - accuracy: 0.5831 - auc: 0.8018 - loss: 0.5914 - val_accuracy: 0.8164 - val_auc: 0.8849 - val_loss: 0.5190
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 50ms/step - accuracy: 0.6015 - auc: 0.8087 - loss: 0.5787 - val_accuracy: 0.7987 - val_auc: 0.8805 - val_loss: 0.5625
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 49ms/step - accuracy: 0.6046 - auc: 0.8183 - loss: 0.5732 - val_accuracy: 0.8075 - val_auc: 0.8895 - val_loss: 0.5531
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 49ms/step - accuracy: 0.6214 - auc: 0.8229 - loss: 0.5587 - val_accuracy: 0.8119 - val_auc: 0.9028 - val_loss: 0.5055
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [5]:
# ==========================================================
# BLOCK 36 & 37: NANO3D_EDGE TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D_Edge.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D_Edge Architecture
# ----------------------------------------------------------
def se_block_3d(x, filters, squeeze_ratio=0.25):
    """Squeeze-and-Excitation (Extremely low parameters)."""
    squeeze_channels = max(1, int(filters * squeeze_ratio))
    se = layers.GlobalAveragePooling3D()(x)
    se = layers.Reshape((1, 1, 1, filters))(se)
    se = layers.Dense(squeeze_channels, activation=tf.nn.swish, use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([x, se])

def nano_edge_block(x, filters, strides=(1,1,1)):
    """Ultra-lightweight factorized block using pseudo-depthwise logic."""
    shortcut = x
    input_filters = x.shape[-1]
    
    # Spatial Pointwise Expansion (1x1x1)
    x = layers.Conv3D(filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Factorized Spatial Filtering (1x3x3) - Using groups to simulate depthwise
    groups_spatial = min(filters, 8) # Fallback to grouped if full depthwise isn't supported efficiently
    
    # Try grouped conv; if it fails (older TF), fall back to standard factorized
    try:
        x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', 
                          groups=groups_spatial, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
        x = layers.Conv3D(filters, (1,3,3), strides=(1, strides[1], strides[2]), padding='same', 
                          use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
                          
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Factorized Temporal Filtering (3x1x1)
    try:
        x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', 
                          groups=groups_spatial, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
         x = layers.Conv3D(filters, (3,1,1), strides=(strides[0], 1, 1), padding='same', 
                          use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
                          
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    
    # Channel Attention
    x = se_block_3d(x, filters)
    
    # Residual matching
    if strides != (1,1,1) or input_filters != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    return x

def create_nano3d_edge_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Micro Stem ---
    x = layers.Conv3D(8, (1,3,3), strides=(1,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(tf.nn.swish)(x)
    x = layers.MaxPooling3D((1,2,2), strides=(1,2,2), padding='same')(x)
    
    # --- Flat Hierarchical Blocks ---
    # Block 1 (8 -> 16 channels)
    b1 = nano_edge_block(x, 16)
    
    # Block 2 (16 -> 24 channels)
    b2 = nano_edge_block(b1, 24, strides=(1,2,2))
    
    # Block 3 (24 -> 32 channels)
    b3 = nano_edge_block(b2, 32, strides=(2,2,2))
    
    # Block 4 (32 -> 48 channels)
    b4 = nano_edge_block(b3, 48, strides=(2,2,2))
    
    # --- Multi-Scale Feature Fusion ---
    # Global average pooling on multi-scale outputs
    pool2 = layers.GlobalAveragePooling3D()(b2)
    pool3 = layers.GlobalAveragePooling3D()(b3)
    pool4 = layers.GlobalAveragePooling3D()(b4)
    
    # Concatenate features (24 + 32 + 48 = 104 parameters fed to final head)
    merged_features = layers.Concatenate()([pool2, pool3, pool4])
    
    # --- Direct Classification Head (No dense layer bottleneck) ---
    x = layers.Dropout(0.4)(merged_features)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D_Edge')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_edge_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_edge_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.4f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D_Edge.

Training Model: Nano3D_Edge...
Epoch 1/50
  3/906 ━━━━━━━━━━━━━━━━━━━━ 30s 34ms/step - accuracy: 0.3333 - auc: 0.6220 - loss: 0.7736   

2026-04-26 19:31:38.995627: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion', 16 bytes spill stores, 16 bytes spill loads
ptxas warning : Registers are spilled to local memory in function 'input_multiply_reduce_fusion_1', 16 bytes spill stores, 16 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 56s 50ms/step - accuracy: 0.5265 - auc: 0.7296 - loss: 0.6273 - val_accuracy: 0.7854 - val_auc: 0.8636 - val_loss: 0.5260
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 44s 48ms/step - accuracy: 0.5607 - auc: 0.7572 - loss: 0.6003 - val_accuracy: 0.7788 - val_auc: 0.8700 - val_loss: 0.5254
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 50ms/step - accuracy: 0.5679 - auc: 0.7740 - loss: 0.5905 - val_accuracy: 0.7743 - val_auc: 0.8660 - val_loss: 0.5154
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 45s 49ms/step - accuracy: 0.5908 - auc: 0.7831 - loss: 0.5833 - val_accuracy: 0.7721 - val_auc: 0.8884 - val_loss: 0.5224
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 50ms/step - accuracy: 0.5958 - auc: 0.8005 - loss: 0.5695 - val_accuracy: 0.7080 - val_auc: 0.8858 - val_loss: 0.5902
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 50ms/step - accuracy: 0.6018 - auc: 0.8011 - loss: 0.5648 - val_accuracy: 0.8031 - val_auc: 0.8861 - val_loss: 0.4885
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [6]:
# ==========================================================
# BLOCK 38 & 39: RESFORMER3D_MAX TRAINING & EVALUATION
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for ResFormer3D_Max.")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
# Assuming compute_class_weight is imported in Block 1
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define ResFormer3D_Max Architecture
# ----------------------------------------------------------
def res_block_3d(x, filters, strides=(1, 1, 1)):
    """Standard robust 3D Residual Block."""
    shortcut = x

    x = layers.Conv3D(filters, (3, 3, 3), strides=strides, padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv3D(filters, (3, 3, 3), strides=(1, 1, 1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

    if strides != (1, 1, 1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1, 1, 1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout=0.3):
    """Standard Transformer Encoder Block."""
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=embed_dim, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Add()([x, inputs])

    y = layers.LayerNormalization(epsilon=1e-6)(x)
    y = layers.Dense(ff_dim, activation=tf.nn.gelu)(y)
    y = layers.Dropout(dropout)(y)
    y = layers.Dense(embed_dim)(y)
    return layers.Add()([y, x])

def create_resformer_max(input_shape):
    video_input = layers.Input(shape=input_shape)

    # --- Deep 3D CNN Backbone ---
    x = layers.Conv3D(64, (5, 5, 5), strides=(1, 2, 2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling3D((1, 3, 3), strides=(1, 2, 2), padding='same')(x)

    x = res_block_3d(x, 64)
    x = res_block_3d(x, 128, strides=(2, 2, 2))
    x = res_block_3d(x, 256, strides=(2, 2, 2))
    x = res_block_3d(x, 512, strides=(2, 2, 2))

    # --- Prepare for Transformer ---
    # We pool spatially but retain the temporal dimension
    x = layers.AveragePooling3D(pool_size=(1, x.shape[2], x.shape[3]))(x)
    
    # Reshape to (Batch, Time, Features) for Attention
    # Note: If NUM_FRAMES=16 and we downsampled time by factor of 8, Time=2
    time_steps = x.shape[1] 
    features = x.shape[-1]
    x = layers.Reshape((time_steps, features))(x)

    # --- Multi-Head Attention Bottleneck ---
    x = transformer_encoder(x, embed_dim=512, num_heads=8, ff_dim=1024, dropout=0.4)
    x = transformer_encoder(x, embed_dim=512, num_heads=8, ff_dim=1024, dropout=0.4)

    # --- Classification Head ---
    x = layers.GlobalAveragePooling1D()(x)
    
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='ResFormer3D_Max')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_resformer_max(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-4, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_resformer_max.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for ResFormer3D_Max.

Training Model: ResFormer3D_Max...
Epoch 1/50


2026-04-26 20:02:30.751883: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_49', 12 bytes spill stores, 12 bytes spill loads

2026-04-26 20:02:30.752153: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_46', 436 bytes spill stores, 436 bytes spill loads

2026-04-26 20:02:31.118410: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4', 444 bytes spill stores, 444 bytes spill loads

2026-04-26 20:02:31.145065: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_49', 112 bytes spill stores, 112 bytes spill loads

2026-04-26 20:02:31.299198: I external/loca

906/906 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.4442 - auc: 0.5939 - loss: 1.2209

2026-04-26 20:03:26.713486: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1003', 96 bytes spill stores, 96 bytes spill loads



906/906 ━━━━━━━━━━━━━━━━━━━━ 66s 54ms/step - accuracy: 0.4625 - auc: 0.6168 - loss: 1.1774 - val_accuracy: 0.7367 - val_auc: 0.8297 - val_loss: 0.9100
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.4884 - auc: 0.6764 - loss: 0.9913 - val_accuracy: 0.7544 - val_auc: 0.8562 - val_loss: 0.7717
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 51ms/step - accuracy: 0.5226 - auc: 0.7171 - loss: 0.8926 - val_accuracy: 0.6858 - val_auc: 0.8456 - val_loss: 0.7951
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5679 - auc: 0.7687 - loss: 0.8174 - val_accuracy: 0.7920 - val_auc: 0.8767 - val_loss: 0.6974
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 46s 51ms/step - accuracy: 0.5728 - auc: 0.7729 - loss: 0.7921 - val_accuracy: 0.7942 - val_auc: 0.8755 - val_loss: 0.6873
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 47s 52ms/step - accuracy: 0.5803 - auc: 0.7930 - loss: 0.7574 - val_accuracy: 0.8119 - val_auc: 0.8834 - val_loss: 0.6894
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━

In [4]:
# ==========================================================
# BLOCK 45 & 46: NANO3D_TURBO TRAINING & EVALUATION (REV 2)
# ==========================================================
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.backend import clear_session
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_recall_fscore_support, 
    confusion_matrix, balanced_accuracy_score
)
import gc
import numpy as np

# Suppress minor TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 1. Clear GPU Memory
# ----------------------------------------------------------
clear_session()
gc.collect()
print("GPU Memory cleared for Nano3D_Turbo (Rev 2).")

# 2. SOTA Training Configuration
# ----------------------------------------------------------
WEIGHT_DECAY = 1e-4  
LABEL_SMOOTHING = 0.1 

if 'train_df' not in locals():
    raise NameError("train_df not found. Please run Block 1 first.")

y_train_labels = train_df['bin_label'].values
class_weights_dict = dict(enumerate(compute_class_weight('balanced', classes=np.unique(y_train_labels), y=y_train_labels)))

# 3. Custom Callback for Time Tracking
# ----------------------------------------------------------
class TimeHistory(Callback):
    def on_train_begin(self, logs={}):
        self.epoch_times = []
        self.step_times = []
        self.train_start_time = time.time()

    def on_epoch_begin(self, epoch, logs={}):
        self.epoch_time_start = time.time()

    def on_epoch_end(self, epoch, logs={}):
        self.epoch_times.append(time.time() - self.epoch_time_start)

    def on_train_batch_begin(self, batch, logs={}):
        self.batch_time_start = time.time()

    def on_train_batch_end(self, batch, logs={}):
        self.step_times.append(time.time() - self.batch_time_start)

    def on_train_end(self, logs={}):
        self.total_train_time = time.time() - self.train_start_time

time_callback = TimeHistory()

# 4. Define Nano3D_Turbo Architecture
# ----------------------------------------------------------
def turbo_block(x, filters, strides=(1,1,1), groups=4):
    """Ultra-fast parallel block utilizing Grouped Convolutions."""
    shortcut = x
    
    # 1x1x1 bottleneck to reduce channel dimensions before the grouped conv
    bottleneck_filters = filters // 2
    x = layers.Conv3D(bottleneck_filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # 3x3x3 Grouped Convolution
    try:
        x = layers.Conv3D(bottleneck_filters, (3,3,3), strides=strides, padding='same', 
                          groups=groups, use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    except:
        group_list = []
        channels_per_group = bottleneck_filters // groups
        splits = tf.split(x, groups, axis=-1)
        for i in range(groups):
            g = layers.Conv3D(channels_per_group, (3,3,3), strides=strides, padding='same', 
                              use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(splits[i])
            group_list.append(g)
        x = layers.Concatenate(axis=-1)(group_list)

    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x) 
    
    # 1x1x1 expansion
    x = layers.Conv3D(filters, (1,1,1), padding='same', use_bias=False, kernel_regularizer=regularizers.l2(1e-5))(x)
    x = layers.BatchNormalization()(x)

    # Residual matching
    if strides != (1,1,1) or shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1,1,1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
        
    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

def create_nano3d_turbo_model(input_shape):
    video_input = layers.Input(shape=input_shape)
    
    # --- Ultra-Fast Stem ---
    # Aggressive downsampling (2,2,2) right at the start
    x = layers.Conv3D(32, (3,3,3), strides=(2,2,2), padding='same', use_bias=False)(video_input)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # --- Straight-Through Hierarchical Blocks ---
    x = turbo_block(x, 64, strides=(1,2,2), groups=4)
    x = turbo_block(x, 128, strides=(2,2,2), groups=8)
    x = turbo_block(x, 256, strides=(2,2,2), groups=16)
    
    # --- Fully Convolutional Head ---
    # No dense layers. We use a 1x1x1 conv to drop channels down to 1, then pool.
    x = layers.Conv3D(128, (1,1,1), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.GlobalAveragePooling3D()(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    return models.Model(inputs=video_input, outputs=output, name='Nano3D_Turbo')

# 5. Initialize & Compile
# ----------------------------------------------------------
input_shape = (NUM_FRAMES,) + FRAME_SIZE + (3,)
model = create_nano3d_turbo_model(input_shape)

total_steps = len(train_df) // BATCH_SIZE * 50
learning_rate_fn = optimizers.schedules.CosineDecay(initial_learning_rate=1e-3, decay_steps=total_steps)
optimizer = optimizers.AdamW(learning_rate=learning_rate_fn, weight_decay=WEIGHT_DECAY)

model.compile(
    optimizer=optimizer,
    loss=BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING), 
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

# 6. Callbacks
# ----------------------------------------------------------
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=0)
checkpoint = ModelCheckpoint('best_nano3d_turbo_model.keras', monitor='val_auc', mode='max', save_best_only=True, verbose=0)

if 'train_generator' not in locals() or 'val_generator' not in locals():
    raise NameError("Generators not found! Run Block 3 first.")

# 7. Start Training
# ----------------------------------------------------------
print(f"\nTraining Model: {model.name}...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50, 
    class_weight=class_weights_dict,
    callbacks=[early_stopping, checkpoint, time_callback],
    verbose=1
)

# 8. SOTA Evaluation & Inference Benchmarking
# ----------------------------------------------------------
print(f"\nEvaluating Model: {model.name}...")
test_generator = SOTAVideoDataGenerator(test_df, PROCESSED_DATA_DIR, batch_size=BATCH_SIZE, augment=False, shuffle=False)

# Warm-up run 
_ = model.predict(test_generator[0][0], verbose=0)

inf_start_time = time.time()
y_pred_prob = model.predict(test_generator, verbose=0).flatten()
inf_end_time = time.time()

total_inference_time = inf_end_time - inf_start_time
num_samples = len(y_pred_prob)
y_pred = (y_pred_prob > 0.5).astype(int)

y_true = test_generator.get_labels().astype(int)
min_len = min(len(y_true), len(y_pred))
y_true, y_pred, y_pred_prob = y_true[:min_len], y_pred[:min_len], y_pred_prob[:min_len]

# Metrics Calculations
acc = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
bal_acc = balanced_accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

spec_normal = tn / (tn + fp) if (tn + fp) > 0 else 0
spec_anomaly = tp / (tp + fn) if (tp + fn) > 0 else 0

total_params = model.count_params()
trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
non_trainable_params = sum([tf.keras.backend.count_params(w) for w in model.non_trainable_weights])
model_size_mb = (total_params * 4) / (1024 * 1024) 

# ==========================================================
# 9. THESIS OUTPUT REPORT
# ==========================================================
print("\n" + "="*60)
print(" 📊 THESIS RESULTS REPORT ".center(60, "="))
print("="*60)
print(f"Model Name                    : {model.name}")
print(f"Total Parameters              : {total_params:,}")
print(f"Trainable Parameters          : {trainable_params:,}")
print(f"Non-trainable Parameters      : {non_trainable_params:,}")
print(f"Model Size (FP32)             : {model_size_mb:.2f} MB")
print("-" * 60)
print(" COMPUTATIONAL EFFICIENCY ".center(60, "-"))
print(f"Training Epochs Completed     : {len(history.history['loss'])}")
print(f"Total Training Time (hours)   : {time_callback.total_train_time / 3600:.4f}")
print(f"Avg Time per Epoch (seconds)  : {np.mean(time_callback.epoch_times):.2f}")
print(f"Avg Time per Step (ms)        : {np.mean(time_callback.step_times) * 1000:.2f}")
print(f"Inference Batch Size          : {BATCH_SIZE}")
print(f"Inference Latency/Sample (ms) : {(total_inference_time / num_samples) * 1000:.2f}")
print(f"Total Throughput (samples/sec): {num_samples / total_inference_time:.2f}")
print("-" * 60)
print(" PREDICTIVE PERFORMANCE ".center(60, "-"))
print(f"Accuracy                      : {acc:.4f}")
print(f"Balanced Accuracy             : {bal_acc:.4f}")
print(f"AUC                           : {auc:.4f}")
print(f"Precision (Normal / Anomaly)  : {precision[0]:.4f} / {precision[1]:.4f}")
print(f"Recall (Normal / Anomaly)     : {recall[0]:.4f} / {recall[1]:.4f}")
print(f"F1-score (Normal / Anomaly)   : {f1[0]:.4f} / {f1[1]:.4f}")
print(f"Specificity (Normal / Anomaly): {spec_normal:.4f} / {spec_anomaly:.4f}")
print(f"Support (Normal / Anomaly)    : {support[0]} / {support[1]}")
print("="*60 + "\n")

GPU Memory cleared for Nano3D_Turbo (Rev 2).


I0000 00:00:1779042490.619398  820900 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13208 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9



Training Model: Nano3D_Turbo...
Epoch 1/50


I0000 00:00:1779042493.541156  821175 service.cc:152] XLA service 0x756d7c002df0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779042493.541175  821175 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-05-18 00:28:13.652160: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779042494.168239  821175 cuda_dnn.cc:529] Loaded cuDNN version 91900


  5/906 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3425 - auc: 0.8813 - loss: 0.6643

I0000 00:00:1779042498.737200  821175 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


906/906 ━━━━━━━━━━━━━━━━━━━━ 50s 46ms/step - accuracy: 0.5712 - auc: 0.7670 - loss: 0.6019 - val_accuracy: 0.7699 - val_auc: 0.8542 - val_loss: 0.5343
Epoch 2/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5941 - auc: 0.7962 - loss: 0.5775 - val_accuracy: 0.7854 - val_auc: 0.8516 - val_loss: 0.5398
Epoch 3/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.6046 - auc: 0.8022 - loss: 0.5673 - val_accuracy: 0.7942 - val_auc: 0.8698 - val_loss: 0.5301
Epoch 4/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.5982 - auc: 0.8221 - loss: 0.5593 - val_accuracy: 0.7810 - val_auc: 0.8642 - val_loss: 0.5227
Epoch 5/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 46ms/step - accuracy: 0.6137 - auc: 0.8299 - loss: 0.5509 - val_accuracy: 0.7544 - val_auc: 0.8825 - val_loss: 0.5376
Epoch 6/50
906/906 ━━━━━━━━━━━━━━━━━━━━ 41s 45ms/step - accuracy: 0.6192 - auc: 0.8383 - loss: 0.5401 - val_accuracy: 0.7412 - val_auc: 0.8751 - val_loss: 0.6349
Epoch 7/50
906/906 ━━━━━━━━━━━━━━━━━━━━